# Plant / Flower Image Classification with CNN

## Objectives
Multi-class **leaf/flower** image classification (TensorFlow flower photos as stable public dataset).

## CNN Architecture
Depthwise feature extraction with Conv2D blocks + dense classifier head.


In [ ]:
# Optional: install dependencies (uncomment if needed)
# !pip install -q numpy pandas matplotlib seaborn scikit-learn tensorflow requests yfinance

import warnings
warnings.filterwarnings("ignore")

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    mean_squared_error, mean_absolute_error, r2_score,
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

sns.set_theme(style="whitegrid")
print("TensorFlow:", tf.__version__)


## Business Context
Agriculture and retail garden centers use vision models for disease detection and inventory.


In [ ]:
# Plant pathology — use tensorflow flowers as stable proxy for leaf disease CNN practice
# tf.keras.utils.get_file can fetch flower photos
import pathlib
dataset_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
import tarfile, urllib.request
data_root = pathlib.Path(keras.utils.get_file('flower_photos', origin=dataset_url, extract=True))
flower_root = list(data_root.glob('*'))[0] if data_root.name != 'flower_photos' else data_root
if not (flower_root / 'daisy').exists():
    flower_root = data_root / 'flower_photos'

IMG_SIZE = (180, 180)
BATCH = 32
train_ds = keras.utils.image_dataset_from_directory(flower_root, validation_split=0.2, subset='training', seed=SEED, image_size=IMG_SIZE, batch_size=BATCH)
val_ds = keras.utils.image_dataset_from_directory(flower_root, validation_split=0.2, subset='validation', seed=SEED, image_size=IMG_SIZE, batch_size=BATCH)
test_ds = val_ds  # use val as holdout for notebook
num_classes = len(train_ds.class_names)
input_shape = IMG_SIZE + (3,)
class_names = train_ds.class_names
print(class_names)


In [ ]:
# EDA
plt.figure(figsize=(10,4))
for images, labels in train_ds.take(1):
    for i in range(8):
        ax = plt.subplot(2, 4, i+1)
        ax.imshow(images[i].numpy().astype("uint8"))
        ax.set_title(class_names[labels[i]])
        ax.axis("off")
plt.show()


In [ ]:
normalization = layers.Rescaling(1./255)
train_scaled = train_ds.map(lambda x,y: (normalization(x), y)).prefetch(tf.data.AUTOTUNE)
val_scaled = val_ds.map(lambda x,y: (normalization(x), y)).prefetch(tf.data.AUTOTUNE)


In [ ]:
model = models.Sequential([
    layers.Input(shape=IMG_SIZE + (3,)),
    layers.Rescaling(1./255),
    layers.Conv2D(32, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation='relu'),
    layers.GlobalAveragePooling2D(),
    layers.Dense(num_classes, activation='softmax'),
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()


In [ ]:
history = model.fit(train_scaled, validation_data=val_scaled, epochs=20, callbacks=[
    callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    callbacks.ModelCheckpoint('cnn_plant_best.keras', save_best_only=True),
])
pd.DataFrame(history.history).plot()
plt.show()


In [ ]:
model.evaluate(val_scaled, verbose=0)
model.save('cnn_plant_final.keras')
for img, lbl in train_scaled.take(1):
    pred = model.predict(img[:2], verbose=0).argmax(axis=1)
    print('Labels:', lbl[:2].numpy(), 'Pred:', pred)


## Deployment Notes

1. **Serving**: Export with `model.export("saved_model")` for TensorFlow Serving, or wrap `predict` in FastAPI/Flask.
2. **Preprocessing**: Always apply the **same** scaler/encoder fitted on training data (`scaler.pkl`).
3. **Monitoring**: Track input drift, latency, and prediction distribution on live traffic.
4. **Retraining**: Schedule periodic retrain when performance drops below SLA.
5. **Security**: Do not log PII; use HTTPS and auth on inference endpoints.
